In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import argparse

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openplaces.api import get_admin, read_entities
from openplaces.geo.vector import get_areas, get_intersection_over_union
from openplaces.recipe import get_recipe_by_id
from openplaces.viz import show_building

In [ ]:
ADMIN3_IDS = {
    'Eastern North Carolina': {
        'US-NC-BA': 'Beaufort',
        'US-NC-BT': 'Bertie',
        'US-NC-BL': 'Bladen',
        'US-NC-BS': 'Brunswick',
        'US-NC-CD': 'Camden',
        'US-NC-CE': 'Carteret',
        'US-NC-CW': 'Chowan',
        'US-NC-CM': 'Columbus',
        'US-NC-CN': 'Craven',
        'US-NC-CU': 'Cumberland',
        'US-NC-CI': 'Currituck',
        'US-NC-DE': 'Dare',
        'US-NC-DP': 'Duplin',
        'US-NC-ED': 'Edgecombe',
        'US-NC-FR': 'Franklin',
        'US-NC-GT': 'Gates',
        'US-NC-GE': 'Greene',
        'US-NC-HL': 'Halifax',
        'US-NC-HT': 'Harnett',
        'US-NC-HD': 'Hertford',
        'US-NC-HO': 'Hoke',
        'US-NC-HE': 'Hyde',
        'US-NC-JH': 'Johnston',
        'US-NC-JN': 'Jones',
        'US-NC-LN': 'Lenoir',
        'US-NC-MR': 'Martin',
        'US-NC-NA': 'Nash',
        'US-NC-NE': 'New Hanover',
        'US-NC-NO': 'Northampton',
        'US-NC-ON': 'Onslow',
        'US-NC-PM': 'Pamlico',
        'US-NC-PU': 'Pasquotank',
        'US-NC-PD': 'Pender',
        'US-NC-PQ': 'Perquimans',
        'US-NC-PI': 'Pitt',
        'US-NC-RB': 'Robeson',
        'US-NC-SP': 'Sampson',
        'US-NC-SC': 'Scotland',
        'US-NC-TY': 'Tyrrell',
        'US-NC-WK': 'Wake',
        'US-NC-WR': 'Warren',
        'US-NC-WI': 'Washington',
        'US-NC-WY': 'Wayne',
        'US-NC-WO': 'Wilson',
    },
    'Coastal Texas': {
        'US-TX-AA': 'Aransas',
        'US-TX-AU': 'Austin',
        'US-TX-BE': 'Bee',
        'US-TX-BI': 'Brazoria',
        'US-TX-BK': 'Brooks',
        'US-TX-CU': 'Calhoun',
        'US-TX-CA': 'Cameron',
        'US-TX-CH': 'Chambers',
        'US-TX-CO': 'Colorado',
        'US-TX-DE': 'DeWitt',
        'US-TX-DU': 'Duval',
        'US-TX-FT': 'Fayette',
        'US-TX-FR': 'Fort Bend',
        'US-TX-GV': 'Galveston',
        'US-TX-GI': 'Goliad',
        'US-TX-HN': 'Hardin',
        'US-TX-RR': 'Harris',
        'US-TX-HG': 'Hidalgo',
        'US-TX-JA': 'Jackson',
        'US-TX-JR': 'Jasper',
        'US-TX-JE': 'Jefferson',
        'US-TX-JG': 'Jim Hogg',
        'US-TX-JW': 'Jim Wells',
        'US-TX-KE': 'Kenedy',
        'US-TX-KL': 'Kleberg',
        'US-TX-LA': 'Lavaca',
        'US-TX-LT': 'Liberty',
        'US-TX-LK': 'Live Oak',
        'US-TX-MD': 'Matagorda',
        'US-TX-NE': 'Newton',
        'US-TX-NU': 'Nueces',
        'US-TX-OR': 'Orange',
        'US-TX-RF': 'Refugio',
        'US-TX-SP': 'San Patricio',
        'US-TX-SR': 'Starr',
        'US-TX-TY': 'Tyler',
        'US-TX-VI': 'Victoria',
        'US-TX-WR': 'Waller',
        'US-TX-WH': 'Washington',
        'US-TX-WB': 'Webb',
        'US-TX-WN': 'Wharton',
        'US-TX-WY': 'Willacy',
    },
}

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest buildings using a recipe')
# parser.add_argument(
#     '--recipe_id',
#     help='Identifier of the recipe (e.g., "US-NC_buildings-cheer-2026")',
# )
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
);

# Set arguments

In [ ]:
ARGS_TEST = (
    # Example 1: Brunswick, NC (hurricane risk case)
    '--admin_ids US-NC-BS '
    # Example 2: Buncombe, NC (fluvial flood risk case)
    # '--admin_ids US-NC-BO'
    # Example 3: Jefferson and Harris, NC (hurricane risk cases)
    # '--admin_ids US-TX-JE US-TX-RR'
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
# Set to True to show examples in notebook
SHOW_EXAMPLES = True

# Read data

In [ ]:
# Future start of loop
admin_id = args.admin_ids[0]

# For visualization purposes
admin3 = get_admin(admin_id, level=3, geom=True)
admin3

In [ ]:
parcels = read_entities('US-NC_parcel-nconemap-2025', admin_id, geom=True)
buildings_fema = read_entities('US_building-fema-2023', admin_id, geom=True)
buildings_nsi = read_entities('US_building-nsi-2022', admin_id, geom=True)
buildings_microsoft = read_entities('US_building-microsoft-v2', admin_id, geom=True)
if admin_id.startswith('US-NC'):
    buildings_nc = read_entities('US-NC_building-ncdps-2023', admin_id, geom=True)

## Drop parcel duplicates

In [ ]:
# Remove parcels that are identical in everything but the `source_deed`
# these seem to be duplicates from records going over multiple pages
mask_duplicates = parcels.drop(columns=['source_deed']).duplicated()
unique_parcels = parcels[~mask_duplicates].copy()

mask_duplicated = unique_parcels['geo_id'].duplicated()
if mask_duplicated.any():
    raise ValueError(
        'Duplicate `geo_id` values found after dropping duplicate parcels.'
    )
else:
    # Dropping solved duplicate issues - use 'geo_id' as 'parcel_id'
    unique_parcels.index = unique_parcels['geo_id'].rename('parcel_id')

## Remap building groups

In [ ]:
CROSSWALK_RECIPE_ID = 'US_building-nsi-2022_purpose-subgroup-remap'

nsi_crosswalk = get_recipe_by_id(CROSSWALK_RECIPE_ID)
nsi_crosswalk = nsi_crosswalk.set_index(nsi_crosswalk.columns[0])
buildings_nsi = buildings_nsi.join(nsi_crosswalk, on=nsi_crosswalk.index.name)

In [ ]:
CROSSWALK_RECIPE_ID = 'US_building-fema-2023_purpose-subgroup-remap'

fema_crosswalk = get_recipe_by_id(CROSSWALK_RECIPE_ID)
fema_crosswalk = fema_crosswalk.set_index(fema_crosswalk.columns[0])
buildings_fema = buildings_fema.join(fema_crosswalk, on=fema_crosswalk.index.name)

## Calculate polygon areas

In [ ]:
buildings_microsoft['m2'] = get_areas(buildings_microsoft, 'm2')
buildings_fema['m2'] = get_areas(buildings_fema, 'm2')
unique_parcels['ha'] = get_areas(unique_parcels, 'ha')

# Link data

### Documentation of algorithms

#### Hesam Soleimani: Fortuna
*Bounding boxes only finished for 1 county in 6 states: not full inventory*
1. Obtain bounding boxes (**BX**) from high-res satellite imagery
    - Train a model on Microsoft building footprint bounding boxes, then predict.
2. Join Microsoft footprints (**MF**) & FEMA (**FF**) footprints with the help of bounding boxes (**BX**)
    - Intersecting **MF**-**FF**:
        -  **MF** and **FF** match (IoU > 0.5) &rarr; Category I: “perfect”, keep
        -  **MF** and **FF** have highest IoU with same **BX** &rarr; Category II: “good”, keep
        -  All **FF**-matched **BX** lie within **MF** &rarr; Category III: “okay”
            - Only one pair > keep **MF**-**FF**-join.
            - Multiple pairs > keep **FF**, drop **MF**.
        -  All F-matched **BX** don’t lie within same **MF** &rarr; keep **MF**, drop **FF** &rarr; Category IV
    - Non-intersecting **FF**:
        - **FF** and **BX** match (two-way centroid spatial join) &rarr; keep **FF** &rarr; Category V
        - Else: drop.
    - Non-intersecting **MF** &rarr; keep **MF** &rarr; Category VI
    - Non-intersecting **BX** &rarr; keep **BX** &rarr; Category VII
3. Join NSI (**NSI**) to merged building footprints (**B**) 
    -  If **B** has matching **MF** and **FF** &rarr; join **NSI** to **B** via either **MF** or **FF**
    -  If **B** is only **FF** &rarr; join **NSI** to **B** via **FF**
    -  If **B** is only **BX** &rarr; join **NSI** to **B** via **BX**

#### Christoph Nolte: openplaces

1. Join Microsoft building footprints (**MF**) to parcels (**PC**)
    - Intersecting **MF** & **PC**
        - **MF** with unique **PC** &rarr; keep
            - Ignore smaller overlaps (<1/6th area of largest)
        - **MF** with duplicate **PC** &rarr; multi-parcel footprint **MF**
   -  **MF** without **PC** &rarr; keep
    - Add both to buildings (**B**)
2. Join National Structure Inventory (**NSI**) to **B**.
   - *tbd*
3. Join FEMA footprints (**FEMA**) to **B**.
   - *tbd*

## Microsoft footprints on parcels

In [ ]:
# Columns to keep in harmonized dataset
FOOTPRINT_PARCEL_COLS = [
    'footprint_id',
    'parcel_id',
    'm2_intersection',
    'iou',
    'm2_intersection_inner',
    'fraction_of_largest',
]

In [ ]:
# how='identity' (left join) is significantly slower than 'intersection'
# (16s vs 4s in US-NC-BS)
# However, it makes sure that parts of footprints *not* on parcels are
# included in the attribution, which is crucial in locations where
# parcel data is patchy (such as UN-NC-BS)
footprints_on_parcels = get_intersection_over_union(
    buildings_microsoft,
    unique_parcels,
    suffixes=('microsoft', 'parcels'),
    how='identity',
    drop_geometries=False,
)
footprints_on_parcels[['m2_intersection', 'iou', 'geometry']]

### Keep unique parcels per footprint

In [ ]:
mask_multiparcel_footprints = footprints_on_parcels.index.get_level_values(
    'footprint_id'
).duplicated(keep=False)

cols = [v for v in FOOTPRINT_PARCEL_COLS if v in footprints_on_parcels]

# Initiate crosswalk (footprint_id > parcel_id)
footprints_unique_parcel = (
    footprints_on_parcels[~mask_multiparcel_footprints]
    .reset_index()
    .set_index('footprint_id')[['parcel_id'] + cols]
)
link = np.where(
    footprints_unique_parcel['parcel_id'].notnull(), 'unique parcel', 'no parcel'
)
footprints_unique_parcel.insert(1, 'link', link)
footprints_unique_parcel

In [ ]:
if SHOW_EXAMPLES:
    # Display example of unambiguous allocation
    footprint_id = (
        footprints_unique_parcel.query('link=="unique parcel"').sample().index[0]
    )
    print('footprint_id:', footprint_id)
    fig, ax = show_building(
        location=buildings_microsoft.loc[[footprint_id]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        # radius=600
        return_fig_ax=True,
    )
    buildings_microsoft.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    ax.set_title('Example: footprint sits on single parcel')

In [ ]:
# Show example of footprint without a parcel (only occurs if parcels are missing)
if SHOW_EXAMPLES:
    footprints_without_parcel = footprints_unique_parcel.query('link=="no parcel"')
    if len(footprints_without_parcel):
        footprint_id = footprints_without_parcel.sample().index[0]
        print('footprint_id:', footprint_id)
        fig, ax = show_building(
            location=buildings_microsoft.loc[[footprint_id]],
            geodatasets={
                'parcels': parcels,
                'buildings_fema': buildings_fema,
                'buildings_nsi': buildings_nsi,
                'buildings_microsoft': buildings_microsoft,
            },
            # radius=600
            return_fig_ax=True,
        )
        buildings_microsoft.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
            ax=ax, color='red'
        )
        ax.set_title('Example: footprint sits on no parcel')

### Keep unique parcel per footprint after removing small neighbors

In [ ]:
MIN_FRACTION_OF_LARGEST = 0.166

# Compute inner buffer (to identify slivers)
multiparcel_footprints = footprints_on_parcels[mask_multiparcel_footprints].copy()
multiparcel_footprints['fraction_of_largest'] = (
    multiparcel_footprints.groupby('footprint_id', group_keys=False)['m2_intersection']
    .apply(lambda x: (x / x.max()))
    .round(3)
)
multiparcel_footprints_without_small_neighbor = multiparcel_footprints.query(
    f'fraction_of_largest > {MIN_FRACTION_OF_LARGEST}'
)
multiparcel_footprints[['iou', 'm2_intersection', 'fraction_of_largest']]

In [ ]:
mask_multiparcel_footprints_to_split = (
    multiparcel_footprints_without_small_neighbor.index.get_level_values(
        'footprint_id'
    ).duplicated(keep=False)
)

cols = [v for v in FOOTPRINT_PARCEL_COLS if v in multiparcel_footprints]

# Add newly resolved unique parcels to crosswalk
footprints_unique_parcel_without_small_neighbor = (
    multiparcel_footprints_without_small_neighbor[~mask_multiparcel_footprints_to_split]
    .reset_index()
    .set_index('footprint_id')[['parcel_id'] + cols]
)
link = np.where(
    footprints_unique_parcel_without_small_neighbor['parcel_id'].notnull(),
    'unique parcel',
    'no parcel',
)
footprints_unique_parcel_without_small_neighbor.insert(
    1, 'link', link + ' (dropping small neighbor)'
)
footprints_unique_parcel = pd.concat(
    [
        footprints_unique_parcel.drop(
            set(footprints_unique_parcel.index)
            & set(footprints_unique_parcel_without_small_neighbor.index)
        ),
        footprints_unique_parcel_without_small_neighbor,
    ]
).sort_index()


footprints_unique_parcel

In [ ]:
# Join original building footprints to uniquely attributed footprints
if 'geometry' not in footprints_unique_parcel:
    footprints_unique_parcel = gpd.GeoDataFrame(
        footprints_unique_parcel.join(buildings_microsoft['geometry']),
        crs=buildings_microsoft.crs,
    )

In [ ]:
if SHOW_EXAMPLES:
    # Display example of unambiguous allocation after buffer
    footprint_id = footprints_unique_parcel_without_small_neighbor.sample().index[0]
    print('footprint_id:', footprint_id)
    fig, ax = show_building(
        location=buildings_microsoft.loc[[footprint_id]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        return_fig_ax=True,
    )
    buildings_microsoft.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    ax.set_title('Example: ignoring small parcel overlap')

### Split remaining non-unique footprints on parcels

In [ ]:
cols = [
    v
    for v in FOOTPRINT_PARCEL_COLS
    if v in multiparcel_footprints_without_small_neighbor
]

multiparcel_footprints_to_split = multiparcel_footprints_without_small_neighbor[
    mask_multiparcel_footprints_to_split
][cols + ['geometry']]
multiparcel_footprints_to_split.insert(0, 'link', 'multi-parcel footprint')
# multiparcel_footprints_to_split.sort_index()

In [ ]:
footprints_on_parcels_resolved = pd.concat(
    [
        footprints_unique_parcel.reset_index().set_index(['footprint_id', 'parcel_id']),
        multiparcel_footprints_to_split,
    ]
).sort_index()
footprints_on_parcels_resolved

In [ ]:
# from openplaces.io import to_parquet
# to_parquet(footprints_on_parcels_resolved, r"C:\Users\chrnolte\Desktop\footprints_resolved.parquet")

In [ ]:
if SHOW_EXAMPLES:
    # FOOTPRINT_ID = '311473bbc80ff9e202de87fe'  # great example
    # FOOTPRINT_ID = 'ffd2c089502b53be0a3ce5ad'  # part on missing parcel
    # footprint_id = FOOTPRINT_ID

    # Display multi-parcel footprint
    footprint_id = multiparcel_footprints_to_split.sample().index[0][0]
    print('footprint_id:', footprint_id)
    fig, ax = show_building(
        location=buildings_microsoft.loc[[footprint_id]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        # radius=600
        return_fig_ax=True,
    )
    buildings_microsoft.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    # multiparcel_footprints_unresolved.loc[footprint_id].to_crs('epsg:3857').boundary.plot(
    # ax=ax, color='white'
    # )
    ax.set_title('Example: multi-parcel footprint')

### Harmonized footprints

In [ ]:
footprints_on_parcels_resolved['link'].value_counts()

## NSI & split Microsoft footprints

In [ ]:
nsi_footprints = gpd.sjoin(
    buildings_nsi[['geometry']],
    footprints_on_parcels_resolved[['geometry']],
    lsuffix='nsi',
    rsuffix='footprint',
    how='left',
)

if nsi_footprints.index.duplicated().any():
    raise ValueError('Duplicate NSI-footprint links.')

In [ ]:
nsi_footprints['has_footprint'] = nsi_footprints['footprint_id'].notnull()

In [ ]:
from openplaces.io import to_parquet

to_parquet(
    nsi_footprints.sample(frac=1), r'C:\Users\chrnolte\Desktop\nsi_footprints.parquet'
)

In [ ]:
nsi_footprints.plot('has_footprint', legend=True, markersize=1)

## NSI & parcels
NSI appears to be mostly based on parcel data with Microsoft footprints

In [ ]:
nsi_parcel_ids = gpd.sjoin(
    buildings_nsi[['geometry']],
    unique_parcels[['geometry']],
    lsuffix='nsi',
    rsuffix='parcel',
    how='left',
).rename(columns={'index_right': 'parcel_id'})['parcel_id']

if nsi_parcel_ids.index.duplicated().any():
    raise ValueError(
        'Duplicate NSI-parcel links, likely due to parcel duplicates. '
        'Clean/inspect parcel data.'
    )

nsi_parcel_ids

## NSI & Microsoft

In [ ]:
nsi_on_microsoft = gpd.sjoin(
    buildings_nsi[
        [
            'purpose_group',
            'purpose_subgroup',
            'construction_type',
            'foundation_type',
            'n_stories',
            'area_sqft',
            'structure_value',
            'geometry',
        ]
    ],
    buildings_microsoft[['geometry']],
    lsuffix='nsi',
    rsuffix='microsoft',
).rename(columns={'index_right': buildings_nsi.index.name})

assert not nsi_on_microsoft.index.duplicated().any()

In [ ]:
# Largest Intersection over Union
# parcel_id_to_show = parcels_microsoft.iloc[0].name[0]
PARCEL_ID_TO_SHOW = '2a658f029b61be2ecad6fc4e'
# PARCEL_ID_TO_SHOW = parcel_id_to_show

In [ ]:
show_building(
    location=parcels.loc[[PARCEL_ID_TO_SHOW]],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
)

## FEMA & Microsoft

In [ ]:
buildings_microsoft_fema = get_intersection_over_union(
    buildings_microsoft, buildings_fema, suffixes=('microsoft', 'fema')
).sort_values('iou', ascending=False)

buildings_microsoft_fema

In [ ]:
mask_lower_iou_duplicates = buildings_microsoft_fema.index.get_level_values(
    'footprint_id'
).duplicated()
buildings_microsoft_fema_unique = buildings_microsoft_fema[~mask_lower_iou_duplicates]
buildings_microsoft_fema_unique

In [ ]:
# footprint_id = buildings_microsoft_fema_unique.iloc[0].name[0]
footprint_id = buildings_microsoft_fema_unique.iloc[-1].name[0]
footprint_id

In [ ]:
show_building(
    location=buildings_microsoft.loc[[footprint_id]],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
)

# Examples of data issues
Mostly from Brunswick, NC: `US-NC-BS`

## NSI
### Nonsense
Seems to come from HAZUS/NSI-2015

In [ ]:
# Issue: three NSI points that do not exist (circular driveway)
# BUILDING_ID_NSI = 542841649  # Source: HAZUS/NSI-2015

# Example: a manufactured home classified as $0 and "steel"
# BUILDING_ID_NSI = 542714824  # Source: HAZUS/NSI-2015

# Example: looks like manufactured or single-family homes
# NSI says: >4 times multi-family (2 units)
# FEMA (correct): 3 times manufactured
# Parcel: Commercial, $0 building value
BUILDING_ID_NSI = 542464539

In [ ]:
show_building(
    location=buildings_nsi.loc[[BUILDING_ID_NSI]],
    geodatasets={
        'parcels': unique_parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
)

### Sources

In [ ]:
NSI_SOURCE = 'Parcel'
NSI_SOURCE = 'ESRI'
NSI_SOURCE = 'HAZUS/NSI-2015'

building_source_sample = buildings_nsi[buildings_nsi['source'].eq(NSI_SOURCE)].sample()
building_source_sample

In [ ]:
# 4 NSI points, 4 footprints, NSI says Multi-Family
building_source_sample = buildings_nsi.loc[[542464539]]

In [ ]:
show_building(
    location=building_source_sample,
    geodatasets={
        'parcels': unique_parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    radius=200,
)

## NSI + parcels

In [ ]:
# US-NC-BS examples

# # Manufactured home + garage
# parcel_sample = unique_parcels.loc[['707bf24ece118135960af518']]
# parcel_sample = unique_parcels.loc[['669dcc515461a67c9b9348a2']]

# # Manufactured home + in-law suite + shed
# parcel_sample = unique_parcels.loc[['2e5247a34c46125b8e985803']]

# # NSI errors
# # 1. Single-family rural residences with garage / add-on
# # NSI thinks it is 2 multi-family buildings with 2 units, doubling value
# parcel_sample = unique_parcels.loc[['3efa0cb6e8ed132f4b438ee4']]
# parcel_sample = unique_parcels.loc[['3efa0cb6e8ed132f4b438ee4']]

# # 2. Single-family rural residences with add-on
# # NSI thinks it is 2 multi-family buildings with 5-10 units
# # Though value not too far off
# parcel_sample = unique_parcels.loc[['5afe3bd881fb46aac3c00e1d']]

# 3. New development: no NSI but parcels
parcel_sample = unique_parcels.loc[['30de585ddb16170187eea5d2']]

In [ ]:
show_building(
    location=parcel_sample,
    geodatasets={
        'parcels': unique_parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    radius=200,
)

## Footprints

### Teardown?

In [ ]:
# Largest footprint. On industrial wasteland. Looks like an error
FOOTPRINT_ID_TO_SHOW = '914d64cb0494f753f32993e5'

fig, ax = show_building(
    location=buildings_microsoft.loc[[FOOTPRINT_ID_TO_SHOW]],
    geodatasets={
        'parcels': parcels,
        # 'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    radius=600,
    return_fig_ax=True,
)
ax.set_title('Erroneous footprints')

## Parcels

### Unallocated

In [ ]:
if SHOW_EXAMPLES:
    # Shed
    FOOTPRINT_ID = '3a9eadb52181f68d19eb8bb7'

    fig, ax = show_building(
        location=buildings_microsoft.loc[[FOOTPRINT_ID]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        return_fig_ax=True,
    )
    buildings_microsoft.loc[[FOOTPRINT_ID]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    ax.set_title('Example: unnallocated (too small - shed)')

### Missing parcel

In [ ]:
if SHOW_EXAMPLES:
    FOOTPRINT_ID = '33cc67632cc7f5f979828fee'

    fig, ax = show_building(
        location=buildings_microsoft.loc[[FOOTPRINT_ID]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        radius=600,
        return_fig_ax=True,
    )
    buildings_microsoft.loc[[FOOTPRINT_ID]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    ax.set_title('Example: missing parcel')

In [ ]:
# Smaller parcel
if SHOW_EXAMPLES:
    BUILDING_ID_FEMA = 1579061
    fig, ax = show_building(
        location=buildings_fema.loc[[BUILDING_ID_FEMA]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        return_fig_ax=True,
    )
    buildings_fema.loc[[BUILDING_ID_FEMA]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    ax.set_title('Example: missing parcel (FEMA footprint)')

### Apparently incorrectly located parcels

In [ ]:
FOOTPRINT_ID = '38dc2290bea7f282bc16fb6c'

if SHOW_EXAMPLES:
    # Display example of unambiguous allocation after buffer
    fig, ax = show_building(
        location=buildings_microsoft.loc[[FOOTPRINT_ID]],
        geodatasets={
            'parcels': parcels,
            'buildings_fema': buildings_fema,
            'buildings_nsi': buildings_nsi,
            'buildings_microsoft': buildings_microsoft,
        },
        # radius=600
        return_fig_ax=True,
    )
    buildings_microsoft.loc[[FOOTPRINT_ID]].to_crs('epsg:3857').boundary.plot(
        ax=ax, color='red'
    )
    # multiparcel_footprints_unresolved.loc[FOOTPRINT_ID].to_crs('epsg:3857').boundary.plot(
    #     ax=ax, color='white'
    # )
    ax.set_title('Example: multi-parcel footprint - but parcel seems wrong')

### Diverse purpose groups

In [ ]:
GROUP_COLUMN = 'purpose_group'

N_MAX_GROUPS = 20

BY_AREA = True

if not BY_AREA:
    top_groups = unique_parcels[GROUP_COLUMN].value_counts().head(N_MAX_GROUPS).index
else:
    top_groups = (
        unique_parcels.groupby(GROUP_COLUMN)['ha']
        .sum()
        .sort_values(ascending=False)
        .head(N_MAX_GROUPS)
        .index
    )
parcels_of_top_groups = unique_parcels[unique_parcels[GROUP_COLUMN].isin(top_groups)]
fig, ax = plt.subplots(figsize=(10, 10))
parcels_of_top_groups.plot(
    GROUP_COLUMN,
    ax=ax,
    cmap='tab20',
    legend=True,
    legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
)
# admin3.boundary.plot(ax=ax, color='black', linewidth=0.3)
ax.axis('off')